**Finetuning Qwen with GRPO and LoRA: Enhancing AI Reasoning Capabilities**

**Introduction**

Finetuning modern language models has become an essential practice for tailoring artificial intelligence to specific tasks and datasets. In this article, we explore the process of finetuning Qwen using the GRPO (Gradient Reward Policy Optimization) framework. The approach leverages efficient techniques such as Low-Rank Adaptation (LoRA) and fast inference with vLLM, alongside custom reward functions that guide the training process. We begin with setting up the environment, installing dependencies, and configuring the model. Then, we dive into preparing the dataset and defining reward functions to ensure the model’s outputs adhere to desired formats and correctness. This comprehensive guide not only explains each step in the finetuning pipeline but also discusses options for saving and deploying the model in various quantization formats.

#### Step 1: Installing Dependencies

This step installs the necessary Python packages required for finetuning Qwen with GRPO. The packages include:
- **unsloth & vllm:** For fast language model loading and inference.
- **triton:** A specific version needed for compatibility.
- **pynvml:** For managing NVIDIA GPU resources.

In [ ]:
%%capture
# Install necessary packages for fast model inference and GPU management.
!pip install unsloth vllm  
!pip install triton==3.1.0  
!pip install -U pynvml

#### Step 2: Loading the Model with LoRA Adapters

In this step, we load the Qwen model using the FastLanguageModel API from unsloth. We also apply a patch for GRPO using LoRA (Low-Rank Adaptation) with specific parameters. This setup allows efficient model finetuning and fast inference.

In [ ]:
import torch
from unsloth import is_bfloat16_supported
from unsloth import FastLanguageModel, PatchFastRL

# Patch the FastLanguageModel to integrate GRPO-specific modifications.
PatchFastRL("GRPO", FastLanguageModel)

# Set maximum sequence length and LoRA rank (controls the adaptation complexity).
max_seq_length = 1024  # Increase if you need longer reasoning traces.
lora_rank = 64         # Larger rank can improve performance but may slow down training.

# Load the Qwen model in 4-bit mode for reduced memory usage and enable fast inference with vLLM.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,          # Set to False if using LoRA in 16-bit precision.
    fast_inference = True,        # Enable vLLM for faster inference.
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.5, # Adjust GPU memory usage to avoid out-of-memory errors.
)

# Wrap the model with PEFT (Parameter-Efficient Fine-Tuning) using LoRA.
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,          # Use a rank greater than 0; common choices include 8, 16, 32, 64, or 128.
    lora_alpha = lora_rank, # A higher lora_alpha value means that the LoRA layers have a greater influence on the model's output, 
                            # while a lower value reduces this influence
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],                                       # Specify target modules; you can remove QKVO if memory is limited.
    use_gradient_checkpointing = "unsloth",  # Enable gradient checkpointing for long context finetuning.
    random_state = 3407,                     # Set a random seed for reproducibility.
)

#### Step 3: Preparing the Dataset

This block loads and processes the GSM8K dataset using Hugging Face's datasets library. The dataset is formatted to include a system prompt and the user question, along with a function to extract a part of the answer (after a hash marker).

In [ ]:
import re
from datasets import load_dataset, Dataset

# Define a system prompt that specifies the desired response format.
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

# XML-based chain-of-thought (CoT) format template.
XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_hash_answer(text: str) -> str | None:
    # Extracts the answer text that follows the "####" marker.
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# Function to load and prepare the GSM8K dataset.
def get_gsm8k_questions(split = "train") -> Dataset:
    # Load the dataset from the 'openai/gsm8k' repository.
    data = load_dataset('openai/gsm8k', 'main')[split]  # type: ignore
    # Map the dataset to include a formatted prompt and the extracted answer.
    data = data.map(lambda x: {  # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    })  # type: ignore
    return data  # type: ignore

# Create the dataset to be used for training.
dataset = get_gsm8k_questions()

#### Step 4: Defining Reward Functions for GRPO

This section defines several reward functions that assess the model’s responses during finetuning. The rewards are based on correctness, integer response, and adherence to the required XML format.

##### Explanation of Reward Functions

1. **correctness_reward_func**  
   - **Purpose:** This function evaluates whether the generated answer (extracted from the XML response) exactly matches the expected answer.  
   - **How it works:**  
     - It first extracts the answer portion from each generated response using the helper function.
     - It compares each extracted answer with the ground truth answer.
     - If they match, the function awards a reward of 2.0; otherwise, the reward is 0.0.
     - Additionally, it prints out details (question, expected answer, generated response, and extracted answer) to help with debugging and analysis.

2. **int_reward_func**  
   - **Purpose:** This function checks whether the generated answer is a simple integer, which can be useful when the expected output is a digit (e.g., a count).  
   - **How it works:**  
     - Similar to the correctness function, it extracts the answer from the XML-formatted response.
     - It then checks if the extracted answer consists only of digits.
     - If the answer is a digit, it awards a reward of 0.5; otherwise, it returns 0.0.

3. **strict_format_reward_func**  
   - **Purpose:** To ensure that the generated response strictly adheres to a specific XML format. This is important when a precise structure is required (e.g., for downstream processing or evaluation).  
   - **How it works:**  
     - It defines a strict regular expression pattern that the response must exactly match, including specific newlines and tag placements for `<reasoning>` and `<answer>`.
     - The function then checks each response against this pattern.
     - If a response perfectly matches the expected format, it awards a reward of 0.5; if not, it awards 0.0.

4. **soft_format_reward_func**  
   - **Purpose:** This function checks if the response follows the expected XML format in a looser, more flexible manner than the strict version.  
   - **How it works:**  
     - It uses a less restrictive regular expression that allows for minor variations while still requiring the presence of both `<reasoning>` and `<answer>` tags.
     - If a response matches the soft format, it awards a reward of 0.5; otherwise, it returns 0.0.
     - This function provides some tolerance for slight deviations from the strict template.

5. **xmlcount_reward_func**  
   - **Purpose:** To assess the overall structure of the XML response by counting the occurrences of key XML tags and applying small penalties for any extra characters that might indicate formatting errors.  
   - **How it works:**  
     - It uses a helper function (`count_xml`) that increments a reward score if the response contains exactly one occurrence of each critical tag (e.g., `<reasoning>`, `</reasoning>`, `<answer>`, and `</answer>`).
     - If the text has extra characters beyond the expected structure (especially after the `</answer>` tag), it slightly reduces the reward score.
     - The final score is returned as the reward for that response.

Each of these reward functions plays a role in guiding the model during training by encouraging responses that are not only correct but also well-formatted and structured as required by the downstream application.

In [ ]:
def extract_xml_answer(text: str) -> str:
    # Extracts the answer portion from the XML formatted text.
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

# Reward function to compare correctness of the generated answer.
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", 
          f"\nExtracted:\n{extracted_responses[0]}")
    # Award a reward if the extracted response matches the expected answer.
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

# Reward function for integer responses.
def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    # Provide a reward if the response is a digit.
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

# Reward function checking for strict XML format.
def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion strictly adheres to the XML format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    # Reward if the response exactly matches the pattern.
    return [0.5 if match else 0.0 for match in matches]

# Reward function checking for a soft XML format (less strict).
def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion follows the expected XML format (soft check)."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

# Helper function to count XML tags and provide a reward score based on counts.
def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001
    return count

# Reward function using XML tag counts to evaluate formatting.
def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

#### Step 5: Configuring and Running the Trainer

This step sets up the GRPO finetuning process with a specific configuration. It defines training hyperparameters, logging, and gradient accumulation parameters. The GRPOTrainer is then instantiated with the model, tokenizer, reward functions, and training dataset, and training is initiated.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Configure GRPO training parameters.
# This configuration sets up the training hyperparameters, optimization settings, and inference acceleration via vLLM.
training_args = GRPOConfig(
    use_vllm = True,                     # Enable vLLM to accelerate inference during training.
    learning_rate = 5e-6,                # Set the learning rate for the optimizer.
    adam_beta1 = 0.9,                    # First beta parameter for the AdamW optimizer.
    adam_beta2 = 0.99,                   # Second beta parameter for the AdamW optimizer.
    weight_decay = 0.1,                  # Weight decay to regularize the model and prevent overfitting.
    warmup_ratio = 0.1,                  # Fraction of steps used for learning rate warmup.
    lr_scheduler_type = "cosine",        # Use cosine annealing for the learning rate scheduler.
    optim = "adamw_8bit",                # Use 8-bit AdamW optimizer for memory efficiency.
    logging_steps = 1,                   # Log training information every step.
    bf16 = is_bfloat16_supported(),      # Use bfloat16 precision if supported by the GPU.
    fp16 = not is_bfloat16_supported(),  # Otherwise, fall back to fp16 precision.
    per_device_train_batch_size = 1,     # Batch size per device during training.
    gradient_accumulation_steps = 1,     # Accumulate gradients over this many steps (increase for smoother training if needed).
    num_generations = 8,                 # Number of generations per prompt (reduce if memory issues occur).
    max_prompt_length = 256,             # Maximum length for the input prompt.
    max_completion_length = 200,         # Maximum length for the generated completion.
    # num_train_epochs = 1,               # Uncomment this line to run training for one epoch.
    max_steps = 250,                     # Maximum number of training steps.
    save_steps = 250,                    # Save the model checkpoint every specified number of steps.
    max_grad_norm = 0.1,                 # Maximum gradient norm for gradient clipping.
    report_to = "none",                  # Disable reporting to external services like WandB.
    output_dir = "outputs",              # Directory to save the training outputs and checkpoints.
)

# Instantiate the GRPO trainer with the model, tokenizer, reward functions, and training dataset.
trainer = GRPOTrainer(
    model = model,                       # The language model to be trained.
    processing_class = tokenizer,        # The tokenizer used to preprocess the data.
    reward_funcs = [
        xmlcount_reward_func,            # Reward function based on XML tag counts.
        soft_format_reward_func,         # Reward function checking for soft adherence to XML formatting.
        strict_format_reward_func,       # Reward function checking for strict XML formatting.
        int_reward_func,                 # Reward function that provides rewards for integer outputs.
        correctness_reward_func,         # Reward function evaluating the correctness of the answer.
    ],
    args = training_args,                # GRPO training configuration.
    train_dataset = dataset,             # The training dataset containing prompts and expected answers.
)

# Begin training using the GRPO algorithm.
trainer.train()

# Save the LoRA-adapted model for later use.
model.save_lora("grpo_saved_lora")

#### Step 6: LLM Inference without LoRA Adapters

This code block shows how to generate a response from the finetuned model without using any LoRA adapters. The model is given a simple user prompt, and inference parameters like temperature and top_p are set for controlled generation.

In [ ]:
# Prepare a text input using the tokenizer's chat template.
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "How many r's are in strawberry?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams

# Set the sampling parameters for text generation.
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)

# Generate a response from the model without applying any LoRA adapter.
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,  # No LoRA adapter is used here.
)[0].outputs[0].text

# Print the generated output.
print(output)

#### Step 7: LLM Inference with LoRA Adapters

Here, the model generates output using the saved LoRA adapter ("grpo_saved_lora"). The input is formatted with a system prompt, and the same sampling parameters are applied for consistency.

In [ ]:
# Prepare a text input with both a system prompt and user question.
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "How many r's are in strawberry?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams

# Set sampling parameters for controlled text generation.
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)

# Generate a response using the saved LoRA adapter.
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),  # Load the saved LoRA adapter.
)[0].outputs[0].text

# Print the generated response.
print(output)

#### Step 8: Merging and Saving the Model

In this final section, there are multiple conditional blocks (set to `False`) that provide options to merge the model weights into different precision formats or save only the LoRA adapters. Additionally, options are provided to save the model in GGUF formats (8-bit, 16-bit, or other quantization methods) for sharing on Hugging Face.

In [ ]:
# Merge to 16bit: Uncomment to merge model weights to 16-bit precision.
if False: 
    model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: 
    model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit: Uncomment to merge model weights to 4-bit precision.
if False: 
    model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: 
    model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Save only the LoRA adapters: Uncomment to save just the LoRA modifications.
if False: 
    model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: 
    model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

# Save to 8bit GGUF format: Uncomment to save the model in GGUF format using Q8_0 quantization.
if False: 
    model.save_pretrained_gguf("model", tokenizer,)
if False: 
    model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF format: Uncomment to save in 16-bit GGUF format.
if False: 
    model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: 
    model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF format: Uncomment to use q4_k_m quantization.
if False: 
    model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: 
    model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF formats at once: Uncomment for simultaneous saving in multiple quantization formats.
if False:
    model.push_to_hub_gguf(
        "hf/model",  # Replace "hf" with your Hugging Face username.
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m"],
        token = "",
    )

#### Conclusion

The GRPO-driven finetuning process for Qwen demonstrates a powerful and flexible approach to enhance AI reasoning and output quality. By integrating techniques like LoRA for parameter-efficient adaptation and employing tailored reward functions, developers can achieve precise control over model behavior and performance. This methodology allows for efficient resource management and faster inference while ensuring the model generates responses in the required format. As the landscape of AI evolves, such finetuning strategies will continue to play a pivotal role in customizing models to meet specific application needs, paving the way for more intelligent, reliable, and task-specific language systems.